In [16]:
import io
import os
import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import random
import tarfile
from transformers import Wav2Vec2Model
from tqdm.notebook import tqdm

device = "cuda"

#df = pd.read_parquet("/home/appleseed/Desktop/frampton/data/emilia-yodas_100.parquet")

In [15]:
class w2v2_clf(nn.Module):
    def __init__(self, freeze_encoder=True, hidden_dim=256):
        super().__init__()
        
        self.encoder = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
        encoder_dim = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(2 * encoder_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        
    def forward(self, x):
        out = self.encoder(
            input_values=x,
            output_hidden_states=False,
            return_dict=True,
        )
        hidden = out.last_hidden_state

        mean = hidden.mean(dim=1)
        std = hidden.std(dim=1)

        pooled = torch.cat([mean, std], dim=-1)
        logits = self.classifier(pooled).squeeze(-1)
        return logits

model = w2v2_clf(freeze_encoder=True, hidden_dim=3072)
ckpt = torch.load("w2v2_clf_head_1697.pth", map_location="cpu")
model.classifier.load_state_dict(ckpt, strict=True)
#model = model.to(device)

/home/appleseed/Desktop/frampton/lib/python3.10/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


<All keys matched successfully>

In [18]:
#model = model.to(device)
model.eval()

w2v2_clf(
  (encoder): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): Wav2V

In [2]:
data_dir = "/home/appleseed/Desktop/frampton/data/emilia-yodas"

tar_idx = {}
for i, tar_file in tqdm(enumerate(sorted(os.listdir(data_dir)))):
    # prepare the tar
    TAR_PATH = f"{data_dir}/{tar_file}"
    with tarfile.open(TAR_PATH, "r:*") as tar:
        members = tar.getmembers()
        for member in members:
            if not member.isfile():
                print(f"{member} error")
                continue
            if member.name.endswith('mp3'):
                tar_idx[member.name] = {
                    "tar": TAR_PATH,
                    "mp3": member.name,
                }            
    #if i>20: break

print(list(tar_idx.keys())[:5])
print(tar_idx[list(tar_idx.keys())[0]])

0it [00:00, ?it/s]

['EN_tKvmUvxYZXI_W000006.mp3', 'EN_tKvmUvxYZXI_W000014.mp3', 'EN_tKETJ-iK8Qo_W000001.mp3', 'EN_tKvmUvxYZXI_W000023.mp3', 'EN_tKvmUvxYZXI_W000005.mp3']
{'tar': '/home/appleseed/Desktop/frampton/data/emilia-yodas/EN-B000000.tar', 'mp3': 'EN_tKvmUvxYZXI_W000006.mp3'}


In [3]:
from functools import lru_cache

@lru_cache(maxsize=128)
def open_tar(path):
    return tarfile.open(path, "r")

def load_mp3(entry):
    tar = open_tar(entry["tar"])
    member = tar.getmember(entry["mp3"])
    with tar.extractfile(member) as f:
        return f.read()

In [136]:
rand_idx = np.random.randint(0, len(df))
#rand_idx = 1051666
rand_itemid = df.iloc[rand_idx]['id']
audio_tar = tar_idx[f"{rand_itemid}.mp3"]
audio_bytes = load_mp3(audio_tar)

waveform, sr = torchaudio.load(audio_bytes)
waveform = waveform.mean(dim=0, keepdim=True)
waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
num_samples = waveform.shape[1]
target_len = 3*16000
if num_samples >= target_len:
    start = random.randint(0, num_samples-target_len)
    waveform = waveform[:, start:start+target_len]
else:
    pad_len = target_len-num_samples
    waveform = F.pad(waveform, (0, pad_len))

from IPython.display import Audio
Audio(np.array(waveform), rate=16000, autoplay=True)

/tmp/ipykernel_332183/2850380216.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  Audio(np.array(waveform), rate=16000, autoplay=True)


In [137]:
with torch.inference_mode():
    logit = model(waveform.to(device))
pred = torch.sigmoid(logit)
prob = pred.item()
prob*100

0.689483992755413

In [113]:
rand_idx

1889350

In [ ]:
1051666, 1889350